# 12 — Testing whether the main findings depend on a linear model

The preceding notebooks use Ridge regression as a deliberately simple linear probe. This makes differences between representations easy to interpret, but it leaves open a practical question: can a nonlinear model extract additional predictive information from the same frozen inputs?

This notebook replaces the Ridge head with XGBoost for ten model specifications chosen before examining any XGBoost result. The sample, feature definitions and five held-out borough groups remain identical to Notebook 07. The comparison therefore isolates the modelling head rather than changing the geographical test.


## Models included

The subset preserves the main scientific contrasts while avoiding an unnecessary repeat of all 41 control-adjusted specifications.

**PTAL**

- location controls only;
- location + DINOv2 aerial representation;
- location + Street View content and coverage;
- location + all representations.

**EPC**

- compact property controls only;
- compact controls + DINOv2;
- compact controls + all representations;
- richer property controls;
- richer controls + TESSERA;
- richer controls + all representations.

XGBoost is treated as a nonlinear robustness check. It does not replace the pre-specified Ridge benchmark or reopen encoder selection.


In [ ]:
# Connect Google Drive and load the packages used by the nonlinear robustness analysis.
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys
import json
import gc
import time
import hashlib
import platform
import subprocess
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import sklearn

from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from importlib.metadata import version, PackageNotFoundError
from packaging.version import Version
try:
    installed_xgb = version('xgboost')
except PackageNotFoundError:
    installed_xgb = '0'
if Version(installed_xgb) < Version('2.0.0'):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'xgboost>=2.0'])

import xgboost
from xgboost import XGBRegressor

FINAL_CODE_DIR = Path('/content/drive/MyDrive/GEOG0105/CODE/FINAL_PIPELINE')
if str(FINAL_CODE_DIR) in sys.path:
    sys.path.remove(str(FINAL_CODE_DIR))
sys.path.insert(0, str(FINAL_CODE_DIR))

import importlib
import config as _config
importlib.invalidate_caches()
_config = importlib.reload(_config)
globals().update({name: getattr(_config, name) for name in dir(_config) if not name.startswith('__')})

pd.set_option('display.max_columns', 180)
pd.set_option('display.width', 240)

print('Python:', platform.python_version())
print('numpy:', np.__version__, 'pandas:', pd.__version__)
print('sklearn:', sklearn.__version__, 'xgboost:', xgboost.__version__)


## 1. Load the frozen data, model definitions and borough folds

Every preceding modelling stage must have passed its integrity and interpretation checks. Model identities and feature-column hashes are compared with Notebook 07 so the XGBoost inputs cannot silently drift from the Ridge comparators.


In [ ]:
# Verify prerequisites and read the canonical modelling table.
required_paths = [
    FINAL_MODEL_TABLE_PATH,
    FEATURE_MANIFEST_JSON_PATH,
    RIDGE_OUTER_FOLDS_PATH,
    INCREMENTAL_RUN_SPEC_PATH,
    INCREMENTAL_AUDIT_PATH,
    INCREMENTAL_RESULTS_PATH,
    INCREMENTAL_PREDICTIONS_PATH,
    RANDOM_CV_AUDIT_PATH,
    IMAGE_LOCATION_AUDIT_PATH,
    PCA64_AUDIT_PATH,
    GEOMETRIC_AUDIT_PATH,
]
for path in required_paths:
    assert path.exists(), f'Missing prerequisite: {path}'

for audit_path in [
    INCREMENTAL_AUDIT_PATH,
    RANDOM_CV_AUDIT_PATH,
    IMAGE_LOCATION_AUDIT_PATH,
    PCA64_AUDIT_PATH,
    GEOMETRIC_AUDIT_PATH,
]:
    audit = json.loads(audit_path.read_text())
    assert audit['integrity_gate_pass'] is True
    assert audit['interpretation_gate_pass'] is True

df = pd.read_parquet(FINAL_MODEL_TABLE_PATH)
manifest = json.loads(FEATURE_MANIFEST_JSON_PATH.read_text())
source_07_spec = json.loads(INCREMENTAL_RUN_SPEC_PATH.read_text())
source_07_audit = json.loads(INCREMENTAL_AUDIT_PATH.read_text())

target_col = manifest['target_column']
group_col = manifest['group_column']
categorical_master = set(manifest['categorical_columns'])
feature_sets = manifest['feature_sets']

assert len(df) == 26597
assert df['sample_id'].is_unique
assert df[target_col].notna().all()
assert df[group_col].notna().all()
assert df['task'].value_counts().to_dict() == {'EPC': 20000, 'PTAL': 6597}

df = df.sort_values(['task', 'sample_id'], kind='mergesort').reset_index(drop=True)
task_counts = df['task'].value_counts().to_dict()

model_key_frame = df[['sample_id', 'task', group_col, target_col]].copy()
model_key_hash = hashlib.sha256(
    pd.util.hash_pandas_object(model_key_frame, index=False).values.tobytes()
).hexdigest()
manifest_hash = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()

assert source_07_spec['model_key_sha256'] == model_key_hash
assert source_07_spec['feature_manifest_sha256'] == manifest_hash
assert source_07_audit['model_key_sha256'] == model_key_hash
assert source_07_audit['feature_manifest_sha256'] == manifest_hash

print('Frozen prerequisite chain: PASS')
print('Samples:', task_counts)


In [ ]:
# Reconstruct the ten pre-selected specifications from the frozen manifest.
SELECTED_MODEL_IDS = [
    'PTAL_spatial_baseline',
    'PTAL_spatial_baseline__plus__DINOv2',
    'PTAL_spatial_baseline__plus__StreetView_CLIP_plus_metadata',
    'PTAL_spatial_baseline__plus__All_representations_plus_SV_metadata',
    'EPC_controls_sparse',
    'EPC_controls_sparse__plus__DINOv2',
    'EPC_controls_sparse__plus__All_representations_plus_SV_metadata',
    'EPC_controls_extensive',
    'EPC_controls_extensive__plus__TESSERA',
    'EPC_controls_extensive__plus__All_representations_plus_SV_metadata',
]

def dedupe(columns):
    return list(dict.fromkeys(columns))

model_features = {
    'PTAL_spatial_baseline': list(feature_sets['PTAL_spatial_baseline']),
    'PTAL_spatial_baseline__plus__DINOv2': dedupe(feature_sets['PTAL_spatial_baseline'] + feature_sets['DINOv2']),
    'PTAL_spatial_baseline__plus__StreetView_CLIP_plus_metadata': dedupe(feature_sets['PTAL_spatial_baseline'] + feature_sets['StreetView_CLIP_plus_metadata']),
    'PTAL_spatial_baseline__plus__All_representations_plus_SV_metadata': dedupe(feature_sets['PTAL_spatial_baseline'] + feature_sets['All_representations_plus_SV_metadata']),
    'EPC_controls_sparse': list(feature_sets['EPC_controls_sparse']),
    'EPC_controls_sparse__plus__DINOv2': dedupe(feature_sets['EPC_controls_sparse'] + feature_sets['DINOv2']),
    'EPC_controls_sparse__plus__All_representations_plus_SV_metadata': dedupe(feature_sets['EPC_controls_sparse'] + feature_sets['All_representations_plus_SV_metadata']),
    'EPC_controls_extensive': list(feature_sets['EPC_controls_extensive']),
    'EPC_controls_extensive__plus__TESSERA': dedupe(feature_sets['EPC_controls_extensive'] + feature_sets['TESSERA']),
    'EPC_controls_extensive__plus__All_representations_plus_SV_metadata': dedupe(feature_sets['EPC_controls_extensive'] + feature_sets['All_representations_plus_SV_metadata']),
}

source_specs = pd.DataFrame(source_07_spec['model_specifications']).set_index('model_id')
assert set(SELECTED_MODEL_IDS).issubset(source_specs.index)

def columns_sha256(columns):
    return hashlib.sha256(json.dumps(list(columns), separators=(',', ':')).encode()).hexdigest()

def optional_text(value):
    return None if pd.isna(value) else str(value)

model_records = []
for model_id in SELECTED_MODEL_IDS:
    source = source_specs.loc[model_id]
    cols = model_features[model_id]
    assert cols and not [c for c in cols if c not in df.columns]
    assert len(cols) == int(source['n_features_manifest'])
    assert columns_sha256(cols) == source['feature_columns_sha256']
    model_records.append({
        'task': source['task'],
        'model_id': model_id,
        'baseline_id': optional_text(source['baseline_id']),
        'control_family': source['control_family'],
        'analysis_role': source['analysis_role'],
        'added_feature_set': optional_text(source['added_feature_set']),
        'n_features_manifest': len(cols),
        'feature_columns_sha256': columns_sha256(cols),
    })

model_specs = pd.DataFrame(model_records)
assert model_specs.groupby('task').size().to_dict() == {'EPC': 6, 'PTAL': 4}
display(model_specs)


## 2. Preserve the Street View and missing-data treatment

Structural Street View absence is handled exactly as in the Ridge analysis. Learned imputation and categorical encoding are fitted inside the relevant training subset. Numeric standardisation is omitted because tree splits are unaffected by feature scale.


In [ ]:
# Apply the same structural Street View coverage rules as Notebook 07.
SV_META_COLS = ['sv_has_streetview', 'sv_n_images', 'sv_min_dist_m', 'sv_mean_dist_m']
SV_CLIP_COLS = feature_sets['StreetView_CLIP_only']

def apply_structural_sv_metadata_fill(task_df, task):
    out = task_df.copy()
    radius = {'PTAL': float(STREETVIEW_PTAL_RADIUS_M), 'EPC': float(STREETVIEW_EPC_RADIUS_M)}[task]
    has = pd.to_numeric(out['sv_has_streetview'], errors='coerce')
    clip_complete = out[SV_CLIP_COLS].notna().all(axis=1)
    clip_all_missing = out[SV_CLIP_COLS].isna().all(axis=1)
    has = has.where(has.notna(), clip_complete.astype(int)).astype(int)
    assert has.isin([0, 1]).all()
    assert clip_complete[has.eq(1)].all()
    assert clip_all_missing[has.eq(0)].all()
    out['sv_has_streetview'] = has
    for col in SV_META_COLS[1:]:
        out[col] = pd.to_numeric(out[col], errors='coerce')
    no_sv = has.eq(0)
    out.loc[no_sv, 'sv_n_images'] = 0.0
    for col in ['sv_min_dist_m', 'sv_mean_dist_m']:
        out.loc[no_sv & out[col].isna(), col] = radius
    return out

def make_onehot():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=True, dtype=np.float32)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=True, dtype=np.float32)

def build_preprocessor(feature_cols):
    categorical_cols = [c for c in feature_cols if c in categorical_master]
    numeric_cols = [c for c in feature_cols if c not in categorical_master]
    transformers = []
    if numeric_cols:
        transformers.append((
            'num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric_cols
        ))
    if categorical_cols:
        transformers.append((
            'cat', Pipeline([
                ('imputer', SimpleImputer(strategy='constant', fill_value='__MISSING__')),
                ('onehot', make_onehot()),
            ]), categorical_cols
        ))
    return ColumnTransformer(transformers, remainder='drop', sparse_threshold=1.0)

def to_float32(matrix):
    return matrix.astype(np.float32, copy=False)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def atomic_csv(frame, path):
    tmp = path.with_name(path.stem + '.tmp' + path.suffix)
    frame.to_csv(tmp, index=False)
    tmp.replace(path)

def atomic_parquet(frame, path):
    tmp = path.with_name(path.stem + '.tmp' + path.suffix)
    frame.to_parquet(tmp, index=False)
    tmp.replace(path)

def atomic_json(obj, path):
    tmp = path.with_name(path.stem + '.tmp' + path.suffix)
    tmp.write_text(json.dumps(obj, indent=2, allow_nan=False))
    tmp.replace(path)

def coerce_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)
    mapped = series.astype(str).str.strip().str.lower().map({
        'true': True, 'false': False, '1': True, '0': False,
    })
    assert mapped.notna().all(), f'Unrecognised Boolean values: {series[mapped.isna()].unique()}'
    return mapped.astype(bool)


## 3. Fix the XGBoost training rule

The tree structure is intentionally conservative. Within each outer training set, complete boroughs form an early-stopping validation subset. The resulting number of boosting rounds is then used to refit the model on the complete outer training set before the held-out boroughs are predicted.

This procedure uses the outer test boroughs only once, for final evaluation. It avoids a large post-hoc hyperparameter search while still allowing model complexity to adapt without observing the test outcome.


In [ ]:
# Detect an available XGBoost device with a tiny smoke test.
def detect_xgb_device():
    X_smoke = np.array([[0.0], [1.0], [2.0], [3.0]], dtype=np.float32)
    y_smoke = np.array([0.0, 1.0, 2.0, 3.0], dtype=np.float32)
    for device in ['cuda', 'cpu']:
        try:
            model = XGBRegressor(
                objective='reg:squarederror', n_estimators=2, max_depth=2,
                tree_method='hist', device=device, n_jobs=1, random_state=RANDOM_STATE,
            )
            model.fit(X_smoke, y_smoke, verbose=False)
            return device
        except Exception:
            continue
    raise RuntimeError('XGBoost could not run on CUDA or CPU.')

XGB_DEVICE = detect_xgb_device()
print('XGBoost device:', XGB_DEVICE)
if XGB_DEVICE != 'cuda':
    print('A T4 GPU runtime is recommended before the full run.')

MAX_BOOST_ROUNDS = 3000
EARLY_STOPPING_ROUNDS = 100
EARLY_VALIDATION_SHARE = 0.20
XGB_FIXED_PARAMS = {
    'objective': 'reg:squarederror',
    'learning_rate': 0.03,
    'max_depth': 4,
    'min_child_weight': 5.0,
    'subsample': 0.80,
    'colsample_bytree': 0.80,
    'reg_alpha': 0.0,
    'reg_lambda': 1.0,
    'gamma': 0.0,
    'max_bin': 256,
    'tree_method': 'hist',
    'eval_metric': 'rmse',
    'n_jobs': 1,
}

def make_xgb(n_estimators, seed, early_stopping=False):
    params = dict(XGB_FIXED_PARAMS)
    params.update({
        'n_estimators': int(n_estimators),
        'random_state': int(seed),
        'device': XGB_DEVICE,
    })
    if early_stopping:
        params['early_stopping_rounds'] = int(EARLY_STOPPING_ROUNDS)
    return XGBRegressor(**params)


In [ ]:
# Reconstruct and verify the exact Notebook-07 borough assignments.
# The frozen hash was calculated from the in-memory table in Notebooks 06/07.
# Reconstructing that table avoids CSV type inference changing an otherwise identical hash.
outer_assignment_rows = []
fold_index_by_task = {}

for task in ['PTAL', 'EPC']:
    task_df = df[df['task'] == task].reset_index(drop=True)
    groups = task_df[group_col].astype(str).to_numpy()
    task_fold = np.full(len(task_df), -1, dtype=int)
    splitter = GroupKFold(n_splits=int(source_07_spec['outer_splits']))

    for fold, (_, test_idx) in enumerate(splitter.split(task_df, task_df[target_col], groups)):
        task_fold[test_idx] = fold
        for idx in test_idx:
            outer_assignment_rows.append({
                'sample_id': task_df.loc[idx, 'sample_id'],
                'task': task,
                'outer_fold': int(fold),
                'borough_code': task_df.loc[idx, group_col],
                'target': float(task_df.loc[idx, target_col]),
            })

    assert (task_fold >= 0).all()
    fold_index_by_task[task] = task_fold

outer_folds = (
    pd.DataFrame(outer_assignment_rows)
    .sort_values(['task', 'sample_id'], kind='mergesort')
    .reset_index(drop=True)
)
saved_outer_folds = (
    pd.read_csv(RIDGE_OUTER_FOLDS_PATH)
    .sort_values(['task', 'sample_id'], kind='mergesort')
    .reset_index(drop=True)
)
pd.testing.assert_frame_equal(
    saved_outer_folds[outer_folds.columns],
    outer_folds,
    check_dtype=False,
    check_exact=False,
    rtol=0,
    atol=1e-12,
)
assert len(outer_folds) == len(df)
assert not outer_folds.duplicated(['task', 'sample_id']).any()
assert outer_folds.groupby(['task', 'borough_code'])['outer_fold'].nunique().eq(1).all()

fold_assignment_hash = hashlib.sha256(
    pd.util.hash_pandas_object(outer_folds, index=False).values.tobytes()
).hexdigest()
assert source_07_spec['outer_fold_assignment_sha256'] == fold_assignment_hash

print('Verified frozen borough-fold SHA256:', fold_assignment_hash)


In [ ]:
# Freeze the exact run specification before fitting any outer test fold.
run_spec = {
    'run_spec_version': '12-xgb-v1-2026-08-23',
    'notebook': '12_selected_xgboost_nonlinearity_robustness.ipynb',
    'analysis_role': 'selected nonlinear robustness; not primary model selection',
    'model_key_sha256': model_key_hash,
    'feature_manifest_sha256': manifest_hash,
    'outer_fold_assignment_sha256': fold_assignment_hash,
    'source_07_run_spec_sha256': source_07_audit['run_spec_sha256'],
    'xgboost_version': xgboost.__version__,
    'device': XGB_DEVICE,
    'selected_models': model_records,
    'outer_splits': 5,
    'early_validation_grouping': 'one deterministic GroupShuffleSplit of outer-training boroughs',
    'early_validation_share': EARLY_VALIDATION_SHARE,
    'early_stopping_rounds': EARLY_STOPPING_ROUNDS,
    'maximum_boost_rounds': MAX_BOOST_ROUNDS,
    'fixed_parameters': XGB_FIXED_PARAMS,
    'refit_rule': 'best boosting round from group-held validation, then refit on full outer training data',
    'preprocessing': {
        'numeric_imputation': 'training-subset median',
        'categorical_imputation': 'training-subset constant __MISSING__',
        'categorical_encoding': 'training-subset OneHotEncoder(handle_unknown=ignore)',
        'numeric_scaling': 'none; tree splits are scale invariant',
        'streetview_structural_absence': 'same rules as Notebook 07',
    },
    'random_state': int(RANDOM_STATE),
}
run_spec_sha256 = hashlib.sha256(json.dumps(run_spec, sort_keys=True).encode()).hexdigest()

if XGBOOST_RUN_SPEC_PATH.exists():
    existing = json.loads(XGBOOST_RUN_SPEC_PATH.read_text())
    assert existing == run_spec, (
        'Existing Notebook-12 checkpoints belong to a different specification or device. '
        'Do not mix them; archive the existing 12 outputs before a documented rerun.'
    )
else:
    atomic_json(run_spec, XGBOOST_RUN_SPEC_PATH)

expected_keys = {
    (row['task'], row['model_id'], fold)
    for row in model_records for fold in range(5)
}
expected_prediction_rows = int(
    4 * task_counts['PTAL'] + 6 * task_counts['EPC']
)
assert len(expected_keys) == 50
assert expected_prediction_rows == 146388

print('Notebook-12 run-spec SHA256:', run_spec_sha256)
print('Expected XGBoost fold-runs:', len(expected_keys))
print('Expected prediction rows:', expected_prediction_rows)


## 4. Fit XGBoost on the frozen outer folds

Each completed fold is saved independently. If Colab disconnects, rerunning the notebook validates completed files and resumes from the first missing fold. Changing runtime device or model settings creates a different run specification and cannot be mixed with existing checkpoints.


In [ ]:
# Validate existing chunks before reusing them.
def checkpoint_is_valid(path, task, model_id, outer_fold, expected_ids):
    if not path.exists():
        return False
    try:
        p = pd.read_parquet(path)
        required = {
            'sample_id', 'task', 'model_id', 'outer_fold', 'borough_code',
            'y_true', 'y_pred', 'run_spec_sha256',
        }
        if not required.issubset(p.columns) or p['sample_id'].duplicated().any():
            return False
        if not p['task'].eq(task).all() or not p['model_id'].eq(model_id).all():
            return False
        if not p['outer_fold'].astype(int).eq(outer_fold).all():
            return False
        if not p['run_spec_sha256'].eq(run_spec_sha256).all():
            return False
        if not np.isfinite(p['y_true']).all() or not np.isfinite(p['y_pred']).all():
            return False
        return set(p['sample_id'].astype(str)) == set(pd.Series(expected_ids).astype(str))
    except Exception:
        return False

if XGBOOST_RESULTS_PATH.exists():
    completed = pd.read_csv(XGBOOST_RESULTS_PATH)
    assert not completed.duplicated(['task', 'model_id', 'outer_fold']).any()
    assert completed['run_spec_sha256'].eq(run_spec_sha256).all()
    result_rows = completed.to_dict('records')
else:
    result_rows = []

for task in ['PTAL', 'EPC']:
    task_df = df[df['task'] == task].reset_index(drop=True)
    task_df = apply_structural_sv_metadata_fill(task_df, task)
    y = pd.to_numeric(task_df[target_col], errors='raise').to_numpy(dtype=np.float32)
    groups = task_df[group_col].astype(str).to_numpy()
    task_fold = fold_index_by_task[task]

    for spec in [r for r in model_records if r['task'] == task]:
        model_id = spec['model_id']
        cols = model_features[model_id]
        X = task_df[cols]

        for outer_fold in range(5):
            test_idx = np.flatnonzero(task_fold == outer_fold)
            train_idx = np.flatnonzero(task_fold != outer_fold)
            run_key = (task, model_id, outer_fold)
            pred_file = XGBOOST_CHUNK_DIR / f'{task}__{model_id}__fold{outer_fold}.parquet'
            existing_keys = {(r['task'], r['model_id'], int(r['outer_fold'])) for r in result_rows}

            if run_key in existing_keys and checkpoint_is_valid(
                pred_file, task, model_id, outer_fold,
                task_df.iloc[test_idx]['sample_id'].to_numpy(),
            ):
                print('SKIP validated checkpoint:', run_key)
                continue

            print('\n' + '=' * 100)
            print(task, '|', model_id, '| held-out borough fold', outer_fold)
            print('=' * 100)

            X_train = X.iloc[train_idx]
            X_test = X.iloc[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            g_train = groups[train_idx]

            inner_splitter = GroupShuffleSplit(
                n_splits=1,
                test_size=EARLY_VALIDATION_SHARE,
                random_state=RANDOM_STATE + outer_fold,
            )
            inner_train, inner_valid = next(inner_splitter.split(X_train, y_train, g_train))
            assert set(g_train[inner_train]).isdisjoint(set(g_train[inner_valid]))

            t0 = time.time()
            pre_early = build_preprocessor(cols)
            Xe_train = to_float32(pre_early.fit_transform(X_train.iloc[inner_train]))
            Xe_valid = to_float32(pre_early.transform(X_train.iloc[inner_valid]))

            early_model = make_xgb(
                MAX_BOOST_ROUNDS,
                seed=RANDOM_STATE + outer_fold,
                early_stopping=True,
            )
            early_model.fit(
                Xe_train, y_train[inner_train],
                eval_set=[(Xe_valid, y_train[inner_valid])],
                verbose=False,
            )
            best_iteration = getattr(early_model, 'best_iteration', None)
            best_rounds = MAX_BOOST_ROUNDS if best_iteration is None else int(best_iteration) + 1
            assert 1 <= best_rounds <= MAX_BOOST_ROUNDS

            del pre_early, Xe_train, Xe_valid, early_model
            gc.collect()

            # Refit the chosen number of trees using every outer-training observation.
            pre_full = build_preprocessor(cols)
            Xf_train = to_float32(pre_full.fit_transform(X_train))
            Xf_test = to_float32(pre_full.transform(X_test))
            final_model = make_xgb(
                best_rounds,
                seed=RANDOM_STATE + outer_fold,
                early_stopping=False,
            )
            final_model.fit(Xf_train, y_train, verbose=False)
            pred = final_model.predict(Xf_test)
            elapsed_s = time.time() - t0
            assert len(pred) == len(test_idx) and np.isfinite(pred).all()

            row = {
                **spec,
                'outer_fold': int(outer_fold),
                'n_train': int(len(train_idx)),
                'n_test': int(len(test_idx)),
                'n_train_boroughs': int(len(np.unique(g_train))),
                'n_test_boroughs': int(len(np.unique(groups[test_idx]))),
                'n_early_train': int(len(inner_train)),
                'n_early_valid': int(len(inner_valid)),
                'n_early_train_boroughs': int(len(np.unique(g_train[inner_train]))),
                'n_early_valid_boroughs': int(len(np.unique(g_train[inner_valid]))),
                'transformed_feature_count': int(Xf_train.shape[1]),
                'best_boost_rounds': int(best_rounds),
                'max_round_boundary': bool(best_rounds == MAX_BOOST_ROUNDS),
                'r2': float(r2_score(y_test, pred)),
                'rmse': rmse(y_test, pred),
                'mae': float(mean_absolute_error(y_test, pred)),
                'fit_seconds': float(elapsed_s),
                'xgb_device': XGB_DEVICE,
                'run_spec_sha256': run_spec_sha256,
            }

            pred_frame = pd.DataFrame({
                'sample_id': task_df.iloc[test_idx]['sample_id'].to_numpy(),
                'task': task,
                'model_id': model_id,
                'outer_fold': int(outer_fold),
                'borough_code': groups[test_idx],
                'y_true': y_test,
                'y_pred': pred,
                'run_spec_sha256': run_spec_sha256,
            })
            atomic_parquet(pred_frame, pred_file)

            result_rows = [
                r for r in result_rows
                if (r['task'], r['model_id'], int(r['outer_fold'])) != run_key
            ]
            result_rows.append(row)
            results_now = pd.DataFrame(result_rows).sort_values(
                ['task', 'model_id', 'outer_fold'], kind='mergesort'
            )
            atomic_csv(results_now, XGBOOST_RESULTS_PATH)
            print(row)

            del pre_full, Xf_train, Xf_test, final_model, pred, pred_frame
            del X_train, X_test
            gc.collect()

print('XGBoost fitting complete or safely checkpointed.')


## 5. Verify every prediction before comparison

The summary is created only after all 50 runs and prediction chunks match the intended task, model, fold and held-out sample IDs. Partial runs remain useful checkpoints but cannot be labelled as final results.


In [ ]:
# Assemble and validate the complete XGBoost prediction set.
results = pd.read_csv(XGBOOST_RESULTS_PATH)
results['outer_fold'] = results['outer_fold'].astype(int)
results['max_round_boundary'] = coerce_bool(results['max_round_boundary'])
actual_keys = set(map(tuple, results[['task', 'model_id', 'outer_fold']].to_numpy()))
missing_keys = sorted(expected_keys - actual_keys)
unexpected_keys = sorted(actual_keys - expected_keys)
print('Completed fold-runs:', len(actual_keys), '/', len(expected_keys))
print('Missing:', len(missing_keys), 'Unexpected:', len(unexpected_keys))
assert not missing_keys, 'Notebook 12 is incomplete: rerun the fitting section.'
assert not unexpected_keys
assert results['run_spec_sha256'].eq(run_spec_sha256).all()

pred_frames = []
for task, model_id, outer_fold in sorted(expected_keys):
    path = XGBOOST_CHUNK_DIR / f'{task}__{model_id}__fold{outer_fold}.parquet'
    task_df = df[df['task'] == task].reset_index(drop=True)
    task_fold = fold_index_by_task[task]
    expected_ids = task_df.loc[task_fold == outer_fold, 'sample_id'].to_numpy()
    assert checkpoint_is_valid(path, task, model_id, outer_fold, expected_ids)
    pred_frames.append(pd.read_parquet(path))

preds = pd.concat(pred_frames, ignore_index=True)
assert len(preds) == expected_prediction_rows
assert not preds.duplicated(['sample_id', 'task', 'model_id', 'outer_fold']).any()
atomic_parquet(preds, XGBOOST_PREDICTIONS_PATH)

print('Validated prediction chunks:', len(pred_frames))
print('Validated prediction rows:', len(preds))


## 6. Compare XGBoost with the matching Ridge model

The two heads use identical outer folds and feature definitions. Fold-level changes are therefore paired by task, model and held-out borough group. They are reported descriptively with mean changes and win counts rather than treating five folds as independent observations for a significance test.


In [ ]:
# Calculate absolute XGBoost performance and exact fold-matched changes from Ridge.
summary_rows = []
for (task, model_id), fold_group in results.groupby(['task', 'model_id']):
    pred_group = preds[(preds['task'] == task) & (preds['model_id'] == model_id)]
    assert len(fold_group) == 5 and len(pred_group) == task_counts[task]
    summary_rows.append({
        'task': task,
        'model_id': model_id,
        'n_features': int(fold_group['n_features_manifest'].iloc[0]),
        'mean_r2': float(fold_group['r2'].mean()),
        'sd_r2': float(fold_group['r2'].std(ddof=1)),
        'mean_rmse': float(fold_group['rmse'].mean()),
        'sd_rmse': float(fold_group['rmse'].std(ddof=1)),
        'mean_mae': float(fold_group['mae'].mean()),
        'sd_mae': float(fold_group['mae'].std(ddof=1)),
        'pooled_r2': float(r2_score(pred_group['y_true'], pred_group['y_pred'])),
        'pooled_rmse': rmse(pred_group['y_true'], pred_group['y_pred']),
        'pooled_mae': float(mean_absolute_error(pred_group['y_true'], pred_group['y_pred'])),
        'median_best_boost_rounds': float(fold_group['best_boost_rounds'].median()),
        'max_round_boundary_hits': int(fold_group['max_round_boundary'].sum()),
        'total_fit_minutes': float(fold_group['fit_seconds'].sum() / 60),
    })

xgb_summary = pd.DataFrame(summary_rows).sort_values(
    ['task', 'mean_r2'], ascending=[True, False], kind='mergesort'
)

ridge_results = pd.read_csv(INCREMENTAL_RESULTS_PATH)
ridge_selected = ridge_results[ridge_results['model_id'].isin(SELECTED_MODEL_IDS)][
    ['task', 'model_id', 'outer_fold', 'r2', 'rmse', 'mae']
].copy()
assert len(ridge_selected) == 50

paired = results[[
    'task', 'model_id', 'outer_fold', 'r2', 'rmse', 'mae', 'best_boost_rounds'
]].merge(
    ridge_selected,
    on=['task', 'model_id', 'outer_fold'],
    how='inner', validate='one_to_one', suffixes=('_xgboost', '_ridge')
)
assert len(paired) == 50
paired['delta_r2_xgboost_minus_ridge'] = paired['r2_xgboost'] - paired['r2_ridge']
paired['delta_rmse_xgboost_minus_ridge'] = paired['rmse_xgboost'] - paired['rmse_ridge']
paired['delta_mae_xgboost_minus_ridge'] = paired['mae_xgboost'] - paired['mae_ridge']
paired['r2_xgboost_win'] = paired['delta_r2_xgboost_minus_ridge'] > 0
paired['rmse_xgboost_win'] = paired['delta_rmse_xgboost_minus_ridge'] < 0
paired['mae_xgboost_win'] = paired['delta_mae_xgboost_minus_ridge'] < 0

ridge_preds = pd.read_parquet(INCREMENTAL_PREDICTIONS_PATH)
ridge_preds = ridge_preds[ridge_preds['model_id'].isin(SELECTED_MODEL_IDS)]
assert len(ridge_preds) == expected_prediction_rows

comparison_rows = []
for (task, model_id), group in paired.groupby(['task', 'model_id']):
    xpred = preds[(preds['task'] == task) & (preds['model_id'] == model_id)]
    rpred = ridge_preds[(ridge_preds['task'] == task) & (ridge_preds['model_id'] == model_id)]
    comparison_rows.append({
        'task': task,
        'model_id': model_id,
        'mean_delta_r2': float(group['delta_r2_xgboost_minus_ridge'].mean()),
        'sd_delta_r2': float(group['delta_r2_xgboost_minus_ridge'].std(ddof=1)),
        'r2_wins_out_of_5': int(group['r2_xgboost_win'].sum()),
        'mean_delta_rmse': float(group['delta_rmse_xgboost_minus_ridge'].mean()),
        'rmse_wins_out_of_5': int(group['rmse_xgboost_win'].sum()),
        'mean_delta_mae': float(group['delta_mae_xgboost_minus_ridge'].mean()),
        'mae_wins_out_of_5': int(group['mae_xgboost_win'].sum()),
        'xgboost_pooled_r2': float(r2_score(xpred['y_true'], xpred['y_pred'])),
        'ridge_pooled_r2': float(r2_score(rpred['y_true'], rpred['y_pred'])),
        'pooled_delta_r2': float(
            r2_score(xpred['y_true'], xpred['y_pred']) - r2_score(rpred['y_true'], rpred['y_pred'])
        ),
        'xgboost_pooled_rmse': rmse(xpred['y_true'], xpred['y_pred']),
        'ridge_pooled_rmse': rmse(rpred['y_true'], rpred['y_pred']),
    })

comparison_summary = pd.DataFrame(comparison_rows).sort_values(
    ['task', 'mean_delta_r2'], ascending=[True, False], kind='mergesort'
)

atomic_csv(xgb_summary, XGBOOST_SUMMARY_PATH)
atomic_csv(paired, XGBOOST_VS_RIDGE_FOLD_PATH)
atomic_csv(comparison_summary, XGBOOST_VS_RIDGE_SUMMARY_PATH)

# Compare every selected XGBoost model with its matching XGBoost baseline.
baseline_fold = results[[
    'task', 'model_id', 'outer_fold', 'r2', 'rmse', 'mae'
]].rename(columns={
    'model_id': 'baseline_id',
    'r2': 'r2_baseline',
    'rmse': 'rmse_baseline',
    'mae': 'mae_baseline',
})
incremental = results[results['baseline_id'].notna()][[
    'task', 'model_id', 'baseline_id', 'analysis_role', 'outer_fold',
    'r2', 'rmse', 'mae',
]].merge(
    baseline_fold,
    on=['task', 'baseline_id', 'outer_fold'],
    how='inner',
    validate='many_to_one',
)
assert len(incremental) == 40
incremental['delta_r2_model_minus_baseline'] = incremental['r2'] - incremental['r2_baseline']
incremental['delta_rmse_model_minus_baseline'] = incremental['rmse'] - incremental['rmse_baseline']
incremental['delta_mae_model_minus_baseline'] = incremental['mae'] - incremental['mae_baseline']
incremental['r2_win'] = incremental['delta_r2_model_minus_baseline'] > 0
incremental['rmse_win'] = incremental['delta_rmse_model_minus_baseline'] < 0
incremental['mae_win'] = incremental['delta_mae_model_minus_baseline'] < 0

incremental_summary_rows = []
for (task, model_id, baseline_id, analysis_role), group in incremental.groupby([
    'task', 'model_id', 'baseline_id', 'analysis_role'
]):
    model_pred = preds[(preds['task'] == task) & (preds['model_id'] == model_id)]
    baseline_pred = preds[(preds['task'] == task) & (preds['model_id'] == baseline_id)]
    assert len(group) == 5
    assert len(model_pred) == len(baseline_pred) == task_counts[task]
    incremental_summary_rows.append({
        'task': task,
        'model_id': model_id,
        'baseline_id': baseline_id,
        'analysis_role': analysis_role,
        'mean_delta_r2': float(group['delta_r2_model_minus_baseline'].mean()),
        'sd_delta_r2': float(group['delta_r2_model_minus_baseline'].std(ddof=1)),
        'r2_wins_out_of_5': int(group['r2_win'].sum()),
        'mean_delta_rmse': float(group['delta_rmse_model_minus_baseline'].mean()),
        'rmse_wins_out_of_5': int(group['rmse_win'].sum()),
        'mean_delta_mae': float(group['delta_mae_model_minus_baseline'].mean()),
        'mae_wins_out_of_5': int(group['mae_win'].sum()),
        'model_pooled_r2': float(r2_score(model_pred['y_true'], model_pred['y_pred'])),
        'baseline_pooled_r2': float(r2_score(baseline_pred['y_true'], baseline_pred['y_pred'])),
        'pooled_delta_r2': float(
            r2_score(model_pred['y_true'], model_pred['y_pred'])
            - r2_score(baseline_pred['y_true'], baseline_pred['y_pred'])
        ),
        'model_pooled_rmse': rmse(model_pred['y_true'], model_pred['y_pred']),
        'baseline_pooled_rmse': rmse(baseline_pred['y_true'], baseline_pred['y_pred']),
    })

incremental_summary = pd.DataFrame(incremental_summary_rows).sort_values(
    ['task', 'mean_delta_r2'], ascending=[True, False], kind='mergesort'
)
assert len(incremental_summary) == 8
atomic_csv(incremental, XGBOOST_INCREMENTAL_FOLD_PATH)
atomic_csv(incremental_summary, XGBOOST_INCREMENTAL_SUMMARY_PATH)

print('XGBoost absolute performance')
display(xgb_summary)
print('XGBoost minus Ridge on identical models and folds')
display(comparison_summary)
print('Incremental value within XGBoost relative to the matching XGBoost baseline')
display(incremental_summary)


## 7. Final completion and interpretation checks

Repeatedly reaching 3,000 boosting rounds would mean the early-stopping ceiling is too low. In that case the predictions remain intact, but interpretation pauses until only the maximum round count is extended. Otherwise the analysis can be frozen after a post-run methodological review.


In [ ]:
# Save the final audit and stop interpretation if the boosting-round ceiling is inadequate.
boundary_counts = results.groupby(['task', 'model_id'])['max_round_boundary'].sum().reset_index()
repeated_boundary = boundary_counts[boundary_counts['max_round_boundary'] >= 3]

audit = {
    'run_spec_path': str(XGBOOST_RUN_SPEC_PATH),
    'run_spec_sha256': run_spec_sha256,
    'model_key_sha256': model_key_hash,
    'feature_manifest_sha256': manifest_hash,
    'outer_fold_assignment_sha256': fold_assignment_hash,
    'source_07_integrity_gate_pass': bool(source_07_audit['integrity_gate_pass']),
    'source_07_interpretation_gate_pass': bool(source_07_audit['interpretation_gate_pass']),
    'xgboost_device': XGB_DEVICE,
    'selected_models': int(len(model_specs)),
    'expected_fold_runs': int(len(expected_keys)),
    'completed_fold_runs': int(len(actual_keys)),
    'expected_prediction_rows': int(expected_prediction_rows),
    'actual_prediction_rows': int(len(preds)),
    'prediction_chunks_validated': int(len(pred_frames)),
    'paired_xgboost_ridge_fold_rows': int(len(paired)),
    'comparison_models': int(len(comparison_summary)),
    'expected_xgboost_incremental_fold_rows': 40,
    'actual_xgboost_incremental_fold_rows': int(len(incremental)),
    'xgboost_incremental_comparison_models': int(len(incremental_summary)),
    'maximum_boost_rounds': int(MAX_BOOST_ROUNDS),
    'models_with_repeated_max_round_hits': int(len(repeated_boundary)),
    'primary_validation': 'same frozen five borough-grouped outer folds as Notebook 07',
    'early_stopping_validation': 'borough-disjoint subset of each outer training fold',
    'inference': 'descriptive fold-paired changes and win counts; no independent-fold p-values',
    'analysis_role': 'nonlinear robustness check; Ridge remains the primary linear probe',
    'integrity_gate_pass': True,
    'interpretation_gate_pass': bool(repeated_boundary.empty),
}
atomic_json(audit, XGBOOST_AUDIT_PATH)

display(pd.Series(audit, name='value'))
if not repeated_boundary.empty:
    display(repeated_boundary)
    raise RuntimeError(
        'Integrity PASS, but interpretation is paused because at least one model reached '
        'the 3,000-round ceiling in three or more folds. Extend only MAX_BOOST_ROUNDS.'
    )

print('Notebook 12 — integrity gate: PASS')
print('Notebook 12 — interpretation gate: PASS')
print('Selected XGBoost robustness analysis complete.')


## What this result adds

XGBoost completed every pre-selected comparison and improved pooled prediction over Ridge for all ten specifications. The substantive ranking remains stable for PTAL and for EPC with compact controls: full fusion ranks above the selected individual representation, and every representation model improves clearly on its matching baseline.

For EPC with richer property controls, the conclusion is more qualified. Under Ridge, full fusion slightly reduced performance; under XGBoost, full fusion and TESSERA each provide a small positive improvement. These gains are much smaller than the improvement produced by adding floor area and construction age. Direct building information therefore remains dominant, while nonlinear interactions recover a modest amount of additional value from representations.

Ridge remains the primary transparent benchmark. XGBoost shows that the main representation findings are not an artefact of linear modelling, while also demonstrating that the estimated incremental value depends partly on the modelling head.
